In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from pypfopt import BlackLittermanModel, risk_models, expected_returns, EfficientFrontier

np.random.seed(42)

if __name__ == "__main__":
   
    bs_price = black_scholes_monte_carlo(S0=100, K=100, T=1, r=0.05, sigma=0.2)
    print(f"Black-Scholes Monte Carlo Price: {bs_price:.2f}")

    
    binomial_price = binomial_option_pricing(S0=100, K=100, T=1, r=0.05, sigma=0.2)
    print(f"Binomial Option Price: {binomial_price:.2f}")

# 1. Black-Scholes Option Pricing %  Binomial Option Pricing Model
def black_scholes_monte_carlo(S0, K, T, r, sigma, simulations=100000):
    Z = np.random.standard_normal(simulations)
    ST = S0 * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
    payoff = np.maximum(ST - K, 0)
    option_price = np.exp(-r * T) * np.mean(payoff)
    return option_price


def binomial_option_pricing(S0, K, T, r, sigma, steps=100):
    dt = T / steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)
    # Initialize asset prices at maturity
    ST = np.array([S0 * (u ** j) * (d ** (steps - j)) for j in range(steps + 1)])
    # Initialize option values at maturity
    option_values = np.maximum(ST - K, 0)
    # Backward induction
    for i in range(steps - 1, -1, -1):
        option_values = disc * (p * option_values[1:] + (1 - p) * option_values[:-1])
    return option_values[0]




    



In [ ]:
# Black-Litterman Model
def black_litterman_model():
    # Sample data
    tickers = ["AAPL", "GOOG", "MSFT", "AMZN"]
    mu = np.array([0.12, 0.10, 0.07, 0.03])  # Expected returns
    sigma = np.array([0.20, 0.25, 0.15, 0.10])  # Standard deviations
    corr_matrix = np.array([
        [1.0, 0.8, 0.6, 0.4],
        [0.8, 1.0, 0.5, 0.3],
        [0.6, 0.5, 1.0, 0.2],
        [0.4, 0.3, 0.2, 1.0]
    ])
    cov_matrix = np.outer(sigma, sigma) * corr_matrix
    market_weights = np.array([0.3, 0.3, 0.2, 0.2])
    # Create DataFrame for PyPortfolioOpt
    cov_df = pd.DataFrame(cov_matrix, index=tickers, columns=tickers)
    market_prior = pd.Series(market_weights, index=tickers)
    # Investor views
    Q = np.array([0.05])  # Expected outperformance
    P = np.array([[1, -1, 0, 0]])  # View: AAPL will outperform GOOG
    omega = np.diag([0.02])  # Uncertainty of views
    # Black-Litterman model
    bl = BlackLittermanModel(cov_df, pi=mu, P=P, Q=Q, omega=omega, market_prior=market_prior)
    ret_bl = bl.bl_returns()
    ef = EfficientFrontier(ret_bl, cov_df)
    weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()
    return cleaned_weights
     bl_weights = black_litterman_model()
    print("Black-Litterman Optimized Weights:")
    print(bl_weights)

# 4. Mean-Variance Optimization
def mean_variance_optimization():
    # Sample data
    tickers = ["AAPL", "GOOG", "MSFT", "AMZN"]
    mu = np.array([0.12, 0.10, 0.07, 0.03])  # Expected returns
    sigma = np.array([0.20, 0.25, 0.15, 0.10])  # Standard deviations
    corr_matrix = np.array([
        [1.0, 0.8, 0.6, 0.4],
        [0.8, 1.0, 0.5, 0.3],
        [0.6, 0.5, 1.0, 0.2],
        [0.4, 0.3, 0.2, 1.0]
    ])
    cov_matrix = np.outer(sigma, sigma) * corr_matrix
    cov_df = pd.DataFrame(cov_matrix, index=tickers, columns=tickers)
    mu_series = pd.Series(mu, index=tickers)
    ef = EfficientFrontier(mu_series, cov_df)
    weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()
    return cleaned_weights

# 5. Value at Risk (VaR) with Monte Carlo Simulation
def monte_carlo_var(S0, mu, sigma, T=1, simulations=100000, confidence_level=0.95):
    Z = np.random.standard_normal(simulations)
    ST = S0 * np.exp((mu - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
    losses = S0 - ST
    var = np.percentile(losses, (1 - confidence_level) * 100)
    return var
mvo_weights = mean_variance_optimization()
    print("Mean-Variance Optimized Weights:")
    print(mvo_weights)

# 6. Conditional Tail Expectation (CTE)
def conditional_tail_expectation(S0, mu, sigma, T=1, simulations=100000, confidence_level=0.95):
    Z = np.random.standard_normal(simulations)
    ST = S0 * np.exp((mu - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
    losses = S0 - ST
    var_threshold = np.percentile(losses, (1 - confidence_level) * 100)
    cte = losses[losses >= var_threshold].mean()
    return cte

# 7. Hull-White Interest Rate Model with Monte Carlo Simulation
def hull_white_model(r0, a, sigma, T=1, dt=0.01, simulations=10000):
    steps = int(T / dt)
    rates = np.zeros((simulations, steps))
    rates[:, 0] = r0
    for t in range(1, steps):
        dr = a * (r0 - rates[:, t - 1]) * dt + sigma * np.sqrt(dt) * np.random.standard_normal(simulations)
        rates[:, t] = rates[:, t - 1] + dr
    return rates


    

    # Value at Risk
    var = monte_carlo_var(S0=1000000, mu=0.07, sigma=0.15)
    print(f"Value at Risk (95% confidence): {var:.2f}")

    # Conditional Tail Expectation
    cte = conditional_tail_expectation(S0=1000000, mu=0.07, sigma=0.15)
    print(f"Conditional Tail Expectation (95% confidence): {cte:.2f}")

    # Hull-White Model Simulation
    rates = hull_white_model(r0=0.03, a=0.1, sigma=0.01)
    # Plotting the first 10 simulated paths
    for i in range(10):
        plt.plot(rates[i])
    plt.title("Hull-White Interest Rate Simulations")
    plt.xlabel("Time Steps")
    plt.ylabel("Interest Rate")
    plt.show()
